# 02 - Data Cleaning & Validation

Notebook này dùng để làm sạch và kiểm tra dữ liệu Sales đã được combine từ `01_data_understanding.ipynb`.

## Mục tiêu
- Load dữ liệu đã combine
- Kiểm tra duplicate
- Phân tích transaction âm
- Phân tích `net_price = 0`
- Xử lý missing `customer_id`
- Chuẩn hóa thời gian
- Tạo business metrics cơ bản
- Tạo validation summary

> **Lưu ý:** Chưa xóa duplicate hay transaction âm cho đến khi xác định được business rule.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Load dữ liệu

Notebook hỗ trợ cả **Parquet** và **Pickle**.

Nếu `pyarrow` cài được, bạn có thể dùng Parquet. Nếu không, notebook sẽ tự fallback sang Pickle.

In [2]:
PROCESSED_PATH = Path("../data/processed")

parquet_path = PROCESSED_PATH / "sales_combined_raw.parquet"
pickle_path = PROCESSED_PATH / "sales_combined_raw.pkl"

if parquet_path.exists():
    sales = pd.read_parquet(parquet_path)
    print("Loaded:", parquet_path.name)

elif pickle_path.exists():
    sales = pd.read_pickle(pickle_path)
    print("Loaded:", pickle_path.name)

else:
    raise FileNotFoundError(
        "Không tìm thấy sales_combined_raw.parquet hoặc sales_combined_raw.pkl "
        "trong ../data/processed/"
    )

print("Shape:", sales.shape)
sales.head()

Loaded: sales_combined_raw.pkl
Shape: (831966, 13)


,month,week,site,branch_id,channel_id,distribution_channel,distribution_channel_code,sold_quantity,cost_price,net_price,customer_id,product_id,source_file
0,2022001,202201,1800,1800,Online,Online,ZF2,1,495720,729000,9847d4248,d77fdd34a14845db97837e059b0aca00TRG42,TT T01-2022_split_1.xlsx
1,2022001,202204,1116,1100,CHTT,Bán lẻ,FP,1,221000,325000,2384aef55,e485c0ab7b9b470cbddb80ea7367e734DEN40,TT T01-2022_split_1.xlsx
2,2022001,202201,1134,1100,CHTT,Bán lẻ,FP,1,255000,375000,20c3e0442,ac88f78262ee4b589bc93b106b67af1dDEN42,TT T01-2022_split_1.xlsx
3,2022001,202204,1612,1600,CHTT,Bán lẻ,FP,1,258400,380000,e8b42ff8f,920641c624934c4a8695347737f8f59dDEN35,TT T01-2022_split_1.xlsx
4,2022001,202202,1511,1500,CHTT,Bán lẻ,FP,1,272000,400000,b8d51499a,6764565f4bb141138af7d9cbf0905d0dHOL33,TT T01-2022_split_1.xlsx


## 2. Tạo bản sao để cleaning

Giữ `sales` làm dữ liệu gốc và dùng `sales_clean` cho các bước xử lý.

In [3]:
sales_clean = sales.copy()

print("Raw shape:", sales.shape)
print("Cleaning shape:", sales_clean.shape)

Raw shape: (831966, 13)
Cleaning shape: (831966, 13)


## 3. Kiểm tra duplicate toàn dataset

In [4]:
duplicate_count = sales_clean.duplicated().sum()

print("Total duplicate rows:", duplicate_count)
print(
    "Duplicate percentage:",
    round(duplicate_count / len(sales_clean) * 100, 4),
    "%"
)

Total duplicate rows: 6119
Duplicate percentage: 0.7355 %


In [5]:
duplicate_rows = sales_clean[
    sales_clean.duplicated(keep=False)
]

duplicate_rows.head(20)

,month,week,site,branch_id,channel_id,distribution_channel,distribution_channel_code,sold_quantity,cost_price,net_price,customer_id,product_id,source_file
4,2022001,202202,1511,1500,CHTT,Bán lẻ,FP,1,272000,400000,b8d51499a,6764565f4bb141138af7d9cbf0905d0dHOL33,TT T01-2022_split_1.xlsx
24,2022001,202202,1205,1200,CHTT,Bán lẻ,FP,1,289000,425000,194ac9bcf,24783093468e490f838568de3b62c0ddHOL34,TT T01-2022_split_1.xlsx
42,2022001,202204,1129,1100,CHTT,Bán lẻ,FP,1,63920,94000,61779b1e7,f53d6dbd8de44f92b2cc586d5b5ac8ebXAM37,TT T01-2022_split_1.xlsx
105,2022001,202204,1136,1100,CHTT,Bán lẻ,FP,1,54400,80000,bc07ddb8b,ea32a403612b448798b57b4a0047eddbXNH41,TT T01-2022_split_1.xlsx
241,2022001,202201,1258,1200,CHTT,Bán lẻ,FP,1,374000,550000,d61e36c2c,4ca23faeefe44fbda0bce6f53c96a6a4DOG31,TT T01-2022_split_1.xlsx
401,2022001,202203,1160,1100,CHTT,Bán lẻ,FP,1,25160,37000,17274d296,bbce3684af87473a973d1ca08e4553bbXAD25,TT T01-2022_split_1.xlsx
456,2022001,202202,1130,1100,CHTT,Bán lẻ,FP,1,40120,59000,8e5c342e2,f8e7202f24314ba89178d6f76cbb47d9XLC32,TT T01-2022_split_1.xlsx
466,2022001,202202,1105,1100,CHTT,Bán lẻ,FP,1,163200,192000,3dd717588,5528edecd7334bdb928e6df49276cb24HOG35,TT T01-2022_split_1.xlsx
513,2022001,202203,1131,1100,CHTT,Bán lẻ,FP,1,699720,1029000,552e3cbc8,4335077df2e24e2e8ca7d41b5c42f990DEN42,TT T01-2022_split_1.xlsx
531,2022001,202204,1503,1500,CHTT,Bán lẻ,FP,1,25160,37000,dfa3b8845,60bbf2727c7d4208b89c2c4d94185f4dTRG25,TT T01-2022_split_1.xlsx


### Duplicate theo business columns

`source_file` là technical metadata dùng cho traceability, không phải business field.

In [6]:
business_columns = [
    col for col in sales_clean.columns
    if col != "source_file"
]

business_duplicate_count = sales_clean.duplicated(
    subset=business_columns
).sum()

print("Business duplicate rows:", business_duplicate_count)

Business duplicate rows: 6119


> **Chưa `drop_duplicates()` tại đây.** Dataset không có `order_id`, nên hai dòng giống nhau chưa chắc là lỗi.

## 4. Phân tích transaction âm

In [7]:
negative_sales = sales_clean[
    (sales_clean["sold_quantity"] < 0) |
    (sales_clean["net_price"] < 0) |
    (sales_clean["cost_price"] < 0)
]

print("Negative transactions:", len(negative_sales))

negative_sales.head(20)

Negative transactions: 26432


,month,week,site,branch_id,channel_id,distribution_channel,distribution_channel_code,sold_quantity,cost_price,net_price,customer_id,product_id,source_file
13,2022001,202204,1124,1100,CHTT,Bán lẻ,FP,-1,-176800,-260000,af943870d,06f7080dde674147aa855475273eccdfXMN25,TT T01-2022_split_1.xlsx
62,2022001,202204,1505,1500,CHTT,Bán lẻ,FP,-1,-255000,-375000,9654e071d,430ca413383e47208abae7124da5b15dDEN40,TT T01-2022_split_1.xlsx
68,2022001,202203,1526,1500,CHTT,Bán lẻ,FP,-1,-197200,-290000,358041268,716ad87de5864a07923ef1c26fb98fd7HOG34,TT T01-2022_split_1.xlsx
92,2022001,202153,1604,1600,CHTT,Bán lẻ,FP,-1,-193800,-285000,555f0d182,74fe13e404434f82afde9b6789795070NAU36,TT T01-2022_split_1.xlsx
109,2022001,202203,1162,1100,CHTT,Bán lẻ,FP,-1,-81600,-120000,990bf2c4a,72b7d8d3afd74c19b1a3ae9afb94f741DEN31,TT T01-2022_split_1.xlsx
152,2022001,202203,1148,1100,CHTT,Bán lẻ,FP,-1,-214200,-315000,0475c4442,1f02c4c801b9454d993fa8f0aa6cdbefHOG32,TT T01-2022_split_1.xlsx
169,2022001,202204,1509,1500,CHTT,Bán lẻ,FP,-1,-645320,-949000,3f7aebc18,8fac3c15c7f943b6bee7bfc946d630feDEN39,TT T01-2022_split_1.xlsx
174,2022001,202204,1504,1500,CHTT,Bán lẻ,FP,-1,-88400,-130000,41fd17be1,55f98434c0c84c72a12e3b2f0324c41fDEN35,TT T01-2022_split_1.xlsx
184,2022001,202203,1508,1500,CHTT,Bán lẻ,FP,-1,-234600,-345000,167a78f6f,839e506226db4e70b54d065fbd1d08afDEN41,TT T01-2022_split_1.xlsx
198,2022001,202204,1108,1100,CHTT,Bán lẻ,FP,-1,-48280,-71000,d2174a5d1,e3e79a5af46d4369b5ecd2a6baf3d5a8CAM33,TT T01-2022_split_1.xlsx


In [8]:
negative_sales[
    ["sold_quantity", "cost_price", "net_price"]
].describe()

,sold_quantity,cost_price,net_price
count,26432.000000,2.643200e+04,2.643200e+04
mean,-1.045816,-2.790102e+05,-3.886401e+05
std,0.592779,2.714514e+05,3.380334e+05
min,-47.000000,-2.184840e+07,-2.302845e+07
25%,-1.000000,-3.196000e+05,-4.600000e+05
50%,-1.000000,-2.142000e+05,-3.090000e+05
75%,-1.000000,-1.335270e+05,-1.900000e+05
max,-1.000000,-1.669100e+04,0.000000e+00


### Kiểm tra dấu của quantity, cost và net price

In [9]:
sign_analysis = pd.DataFrame({
    "qty_negative": sales_clean["sold_quantity"] < 0,
    "cost_negative": sales_clean["cost_price"] < 0,
    "net_negative": sales_clean["net_price"] < 0
})

sign_analysis.value_counts()

qty_negative  cost_negative  net_negative
False         False          False           805534
True          True           True             26005
                             False              427
Name: count, dtype: int64

## 5. Tìm transaction đối ứng của một dòng âm

In [10]:
negative_sales[
    [
        "month",
        "week",
        "customer_id",
        "product_id",
        "sold_quantity",
        "cost_price",
        "net_price"
    ]
].head(10)

,month,week,customer_id,product_id,sold_quantity,cost_price,net_price
13,2022001,202204,af943870d,06f7080dde674147aa855475273eccdfXMN25,-1,-176800,-260000
62,2022001,202204,9654e071d,430ca413383e47208abae7124da5b15dDEN40,-1,-255000,-375000
68,2022001,202203,358041268,716ad87de5864a07923ef1c26fb98fd7HOG34,-1,-197200,-290000
92,2022001,202153,555f0d182,74fe13e404434f82afde9b6789795070NAU36,-1,-193800,-285000
109,2022001,202203,990bf2c4a,72b7d8d3afd74c19b1a3ae9afb94f741DEN31,-1,-81600,-120000
152,2022001,202203,0475c4442,1f02c4c801b9454d993fa8f0aa6cdbefHOG32,-1,-214200,-315000
169,2022001,202204,3f7aebc18,8fac3c15c7f943b6bee7bfc946d630feDEN39,-1,-645320,-949000
174,2022001,202204,41fd17be1,55f98434c0c84c72a12e3b2f0324c41fDEN35,-1,-88400,-130000
184,2022001,202203,167a78f6f,839e506226db4e70b54d065fbd1d08afDEN41,-1,-234600,-345000
198,2022001,202204,d2174a5d1,e3e79a5af46d4369b5ecd2a6baf3d5a8CAM33,-1,-48280,-71000


In [11]:
if len(negative_sales) > 0:
    sample_product = negative_sales.iloc[0]["product_id"]
    sample_customer = negative_sales.iloc[0]["customer_id"]

    opposite_check = sales_clean[
        (sales_clean["product_id"] == sample_product) &
        (sales_clean["customer_id"] == sample_customer)
    ][
        [
            "month",
            "week",
            "sold_quantity",
            "cost_price",
            "net_price",
            "source_file"
        ]
    ].sort_values("week")

    display(opposite_check)
else:
    print("Không có transaction âm.")

,month,week,sold_quantity,cost_price,net_price,source_file
61124,2022001,202203,1,176800,260000,TT T01-2022_split_1.xlsx
13,2022001,202204,-1,-176800,-260000,TT T01-2022_split_1.xlsx
16268,2022001,202204,1,176800,260000,TT T01-2022_split_1.xlsx
59464,2022001,202204,1,176800,260000,TT T01-2022_split_1.xlsx
226521,2022003,202212,1,173586,255000,TT T03-2022_split_1.xlsx


## 6. Phân tích `net_price = 0`

In [12]:
zero_net = sales_clean[
    sales_clean["net_price"] == 0
]

print("Zero net price rows:", len(zero_net))

zero_net.head(20)

Zero net price rows: 1819


,month,week,site,branch_id,channel_id,distribution_channel,distribution_channel_code,sold_quantity,cost_price,net_price,customer_id,product_id,source_file
395,2022001,202202,1800,1800,Online,Online,FP,1,374000,0,5143d0eee,f4286e4700a447eab547e095a91b7160XAM34,TT T01-2022_split_1.xlsx
1378,2022001,202204,1800,1800,Online,Online,FP,-1,-272000,0,5143d0eee,4cace668be104746a27764f50163ac6cTIM31,TT T01-2022_split_1.xlsx
1468,2022001,202201,1148,1100,CHTT,Bán lẻ,FP,1,370600,0,0475c4442,d2d3862579d647d5b8068efa75e7507aDOO44,TT T01-2022_split_1.xlsx
1947,2022001,202202,1800,1800,Online,Online,FP,-1,-584120,0,5143d0eee,c9ccfc81aa504680908187dacbc77789XAM39,TT T01-2022_split_1.xlsx
2032,2022001,202204,1631,1600,CHTT,Bán lẻ,FP,1,506600,0,4efe110d1,2d92da88acb642fdb98ae3cc0cb3fe1aDEN44,TT T01-2022_split_1.xlsx
2165,2022001,202202,1800,1800,Online,Online,FP,-1,-495720,0,5143d0eee,604ea291b65b453189804b00ab555ae2XMN40,TT T01-2022_split_1.xlsx
2962,2022001,202204,1800,1800,Online,Online,FP,1,597720,0,5143d0eee,6b0a1678a06d4018a8642ad576ecd797TRG37,TT T01-2022_split_1.xlsx
3439,2022001,202202,1532,1500,CHTT,Bán lẻ,FP,1,584350,0,3ee8e1745,4859c772d9684395b5fa08cf70b150c9XNH45,TT T01-2022_split_1.xlsx
3688,2022001,202203,1203,1200,CHTT,Bán lẻ,FP,1,462400,0,f763fca95,2bf3e402837c4b3d8f46b03b0a71d0fcKEM45,TT T01-2022_split_1.xlsx
3709,2022001,202202,1219,1200,CHTT,Bán lẻ,FP,1,584350,0,67af1f4d4,9968a78c3bcc4a798c90ab77b73c9beaXNH39,TT T01-2022_split_1.xlsx


In [13]:
if len(zero_net) > 0:
    display(
        zero_net[
            ["sold_quantity", "cost_price", "net_price"]
        ].describe()
    )

,sold_quantity,cost_price,net_price
count,1819.000000,1.819000e+03,1819.0
mean,0.601429,9.809220e+04,0.0
std,1.510877,4.656000e+05,0.0
min,-11.000000,-3.782163e+06,0.0
25%,1.000000,4.216000e+04,0.0
50%,1.000000,8.840000e+04,0.0
75%,1.000000,3.865610e+05,0.0
max,33.000000,2.123550e+06,0.0


In [14]:
zero_net["distribution_channel"].value_counts(dropna=False)

distribution_channel
Online       933
Bán lẻ       879
Phát sinh      7
Name: count, dtype: int64

## 7. Kiểm tra missing `customer_id`

In [15]:
missing_customer_rows = sales_clean[
    sales_clean["customer_id"].isna()
]

print("Missing customer_id:", len(missing_customer_rows))

missing_customer_rows

Missing customer_id: 1


,month,week,site,branch_id,channel_id,distribution_channel,distribution_channel_code,sold_quantity,cost_price,net_price,customer_id,product_id,source_file
626147,2022008,202233,1100,1100,TGPP,Phát sinh,ZF2,1,133528,153164,NaN,798e12cad3444ff689313c5c3da6a356XAD42,TT T08-2022_split_1.xlsx


> **Chưa fill ngay.** Nếu sau này xác nhận transaction vẫn hợp lệ, có thể dùng `"UNKNOWN"` để giữ lại record.

## 8. Kiểm tra mapping Channel

In [16]:
channel_mapping = (
    sales_clean[
        [
            "channel_id",
            "distribution_channel",
            "distribution_channel_code"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "channel_id",
            "distribution_channel"
        ]
    )
)

channel_mapping

,channel_id,distribution_channel,distribution_channel_code
16,CHTT,-,ZF2
219,CHTT,-,ZSF
583795,CHTT,-,ZRD
1,CHTT,Bán lẻ,FP
15,CHTT,Bán sỉ,FP
81961,ONLINE,Online,ZF2
81968,ONLINE,Online,FP
83003,ONLINE,Online,ZRD
647666,ONLINE,Online,ZSF
0,Online,Online,ZF2


## 9. Chuẩn hóa thời gian

Cột `month` dạng `2022001`, `2022002`, ... sẽ được tách thành `year`, `month_number` và `month_date`.

In [17]:
sales_clean["year"] = (
    sales_clean["month"]
    .astype(str)
    .str[:4]
    .astype(int)
)

sales_clean["month_number"] = (
    sales_clean["month"]
    .astype(str)
    .str[-3:]
    .astype(int)
)

sales_clean[
    ["month", "year", "month_number"]
].drop_duplicates().sort_values(
    ["year", "month_number"]
)

,month,year,month_number
0,2022001,2022,1
139807,2022002,2022,2
202335,2022003,2022,3
261003,2022004,2022,4
349996,2022005,2022,5
416003,2022006,2022,6
484235,2022007,2022,7
581704,2022008,2022,8
647000,2022009,2022,9
685301,2022010,2022,10


In [18]:
sales_clean["month_date"] = pd.to_datetime(
    dict(
        year=sales_clean["year"],
        month=sales_clean["month_number"],
        day=1
    )
)

sales_clean[
    ["month", "month_date"]
].drop_duplicates().sort_values("month_date")

,month,month_date
0,2022001,2022-01-01
139807,2022002,2022-02-01
202335,2022003,2022-03-01
261003,2022004,2022-04-01
349996,2022005,2022-05-01
416003,2022006,2022-06-01
484235,2022007,2022-07-01
581704,2022008,2022-08-01
647000,2022009,2022-09-01
685301,2022010,2022-10-01


## 10. Tạo business metrics cơ bản

Ở bước này chỉ tạo `profit` và `profit_margin`.

> Chưa đổi tên `net_price` thành `revenue` cho đến khi xác nhận rõ business definition của dataset.

In [19]:
sales_clean["profit"] = (
    sales_clean["net_price"]
    - sales_clean["cost_price"]
)

sales_clean["profit_margin"] = np.where(
    sales_clean["net_price"] != 0,
    sales_clean["profit"] / sales_clean["net_price"],
    np.nan
)

In [20]:
sales_clean[
    [
        "net_price",
        "cost_price",
        "profit",
        "profit_margin"
    ]
].describe()

,net_price,cost_price,profit,profit_margin
count,8.319660e+05,8.319660e+05,8.319660e+05,830147.000000
mean,3.993313e+05,3.055058e+05,9.382547e+04,0.164430
std,6.886876e+05,6.034099e+05,1.223344e+05,13.575332
min,-2.302845e+07,-2.184840e+07,-6.506482e+06,-2952.535354
25%,1.594260e+05,1.190000e+05,3.232000e+04,0.240930
50%,2.900000e+05,2.069680e+05,7.360000e+04,0.319739
75%,4.590000e+05,3.271420e+05,1.280000e+05,0.320000
max,7.424144e+07,6.082435e+07,1.503360e+07,0.514212


## 11. Kiểm tra transaction lỗ

In [21]:
loss_transactions = sales_clean[
    sales_clean["profit"] < 0
]

print("Loss transactions:", len(loss_transactions))

loss_transactions[
    [
        "product_id",
        "sold_quantity",
        "cost_price",
        "net_price",
        "profit"
    ]
].head(20)

Loss transactions: 35697


,product_id,sold_quantity,cost_price,net_price,profit
10,9ff5be5a5c4b40b5a2655d05b0954798DEN28,1,156400,138000,-18400
13,06f7080dde674147aa855475273eccdfXMN25,-1,-176800,-260000,-83200
46,d82b5c2b0f6443fbad576b880be41aa1DEN39,1,390000,360000,-30000
62,430ca413383e47208abae7124da5b15dDEN40,-1,-255000,-375000,-120000
68,716ad87de5864a07923ef1c26fb98fd7HOG34,-1,-197200,-290000,-92800
92,74fe13e404434f82afde9b6789795070NAU36,-1,-193800,-285000,-91200
107,d4979c8423a840dfb8eb9ccf68a394a8XND29,1,210800,69000,-141800
109,72b7d8d3afd74c19b1a3ae9afb94f741DEN31,-1,-81600,-120000,-38400
152,1f02c4c801b9454d993fa8f0aa6cdbefHOG32,-1,-214200,-315000,-100800
169,8fac3c15c7f943b6bee7bfc946d630feDEN39,-1,-645320,-949000,-303680


## 12. Validation Summary

In [22]:
validation = {
    "rows": len(sales_clean),

    "missing_customer":
        sales_clean["customer_id"].isna().sum(),

    "negative_quantity":
        (sales_clean["sold_quantity"] < 0).sum(),

    "negative_cost":
        (sales_clean["cost_price"] < 0).sum(),

    "negative_net":
        (sales_clean["net_price"] < 0).sum(),

    "zero_net":
        (sales_clean["net_price"] == 0).sum(),

    "business_duplicates":
        sales_clean.duplicated(
            subset=business_columns
        ).sum(),

    "unique_products":
        sales_clean["product_id"].nunique(),

    "unique_customers":
        sales_clean["customer_id"].nunique()
}

validation_summary = pd.Series(validation, name="value")

validation_summary

rows                   831966
missing_customer            1
negative_quantity       26432
negative_cost           26432
negative_net            26005
zero_net                 1819
business_duplicates      6119
unique_products         30367
unique_customers         1173
Name: value, dtype: int64

## 13. Data Quality Table

In [23]:
quality_report = pd.DataFrame({
    "dtype": sales_clean.dtypes.astype(str),
    "missing_count": sales_clean.isna().sum(),
    "missing_pct": (sales_clean.isna().mean() * 100).round(4),
    "unique_values": sales_clean.nunique()
})

quality_report

,dtype,missing_count,missing_pct,unique_values
month,int64,0,0.0000,19
week,int64,0,0.0000,85
site,int64,0,0.0000,227
branch_id,int64,0,0.0000,8
channel_id,str,0,0.0000,5
distribution_channel,str,0,0.0000,7
distribution_channel_code,str,0,0.0000,9
sold_quantity,int64,0,0.0000,124
cost_price,int64,0,0.0000,6314
net_price,int64,0,0.0000,25763


## 14. Dừng tại đây trước khi cleaning chính thức

Chưa thực hiện:

- `drop_duplicates()`
- `dropna()`
- loại transaction âm
- loại `net_price = 0`

Sau khi chạy notebook này, cần xem các output để thiết kế **Cleaning Rules có business justification**.